In [1]:
import requests
from bs4 import BeautifulSoup

url = "http://books.toscrape.com/"
response = requests.get(url)
soup = BeautifulSoup(response.text, "html.parser")

In [27]:
titles = []
for h3_tag in soup.find_all('h3'):
    a_tag = h3_tag.find('a')
    if a_tag and 'title' in a_tag.attrs:
        titles.append(a_tag['title'])

display(titles)

[]

In [29]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time # Import time for adding delays
from urllib.parse import urljoin # For robust URL construction

all_books_data = [] # Changed to store dictionaries of book data (title, category, price)
base_url = "http://books.toscrape.com/catalogue/"
current_page_suffix = "page-1.html" # Starting from the first page of the catalogue

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

# Loop until 100 books are collected or no more pages are found
while len(all_books_data) < 100:
    full_catalogue_url = base_url + current_page_suffix
    print(f"Scraping catalogue page: {full_catalogue_url}")
    try:
        response = requests.get(full_catalogue_url, headers=headers)
        response.raise_for_status() # Raise an exception for HTTP errors (e.g., 404, 403)
        soup = BeautifulSoup(response.text, "html.parser")

        # Extract book information from the current catalogue page
        for article in soup.find_all('article', class_='product_pod'):
            h3_tag = article.find('h3')
            a_tag = h3_tag.find('a')

            if a_tag and 'title' in a_tag.attrs:
                book_title = a_tag['title']
                book_relative_url = a_tag['href']

                # Construct the full URL for the book's detail page using urljoin for robustness
                book_detail_url = urljoin(full_catalogue_url, book_relative_url)

                print(f"  Scraping book detail: {book_detail_url}")
                book_category = 'N/A'
                book_price = 'N/A'

                try:
                    book_response = requests.get(book_detail_url, headers=headers)
                    book_response.raise_for_status()
                    book_soup = BeautifulSoup(book_response.text, "html.parser")

                    # Extract category from the breadcrumbs on the detail page
                    breadcrumb = book_soup.find('ul', class_='breadcrumb')
                    if breadcrumb:
                        # The category is usually the third <li> (index 2) in the breadcrumb list
                        # Example: Home > Category > Book Title
                        category_li_tags = breadcrumb.find_all('li')
                        if len(category_li_tags) >= 3:
                            category_a_tag = category_li_tags[2].find('a')
                            if category_a_tag:
                                book_category = category_a_tag.text.strip()

                    # Extract price
                    price_tag = book_soup.find('p', class_='price_color')
                    if price_tag:
                        price_text = price_tag.text.strip()
                        # Clean price string (remove currency symbol '£' and any non-numeric characters)
                        # 'Â£' can sometimes appear, so replace both 'Â' and '£'
                        book_price = float(price_text.replace('£', '').replace('Â', ''))

                    all_books_data.append({'Book Title': book_title, 'Category': book_category, 'Price': book_price})

                    if len(all_books_data) >= 100:
                        break # Stop if we have collected 100 books

                    time.sleep(0.1) # Be polite: add a small delay between book detail requests

                except requests.exceptions.RequestException as e:
                    print(f"  Failed to scrape book detail {book_detail_url}: {e}")
                    # If detail scraping fails, still add the title with N/A for category/price
                    all_books_data.append({'Book Title': book_title, 'Category': 'N/A', 'Price': 'N/A'})
                except Exception as e: # Catch other potential errors during parsing
                    print(f"  Error parsing book detail {book_detail_url}: {e}")
                    # If parsing fails, still add the title with N/A for category/price
                    all_books_data.append({'Book Title': book_title, 'Category': 'N/A', 'Price': 'N/A'})


        if len(all_books_data) >= 100: # Check again in case inner loop broke
            break

        # Find the link to the next page
        next_button = soup.find('li', class_='next')
        if next_button:
            # The 'href' attribute contains the suffix for the next page
            current_page_suffix = next_button.find('a')['href']
            time.sleep(0.5) # Add a delay between catalogue page requests
        else:
            print("No more pages found.")
            break # Exit loop if there's no 'next' button

    except requests.exceptions.RequestException as e:
        print(f"Request failed for {full_catalogue_url}: {e}")
        break # Exit loop if any request error occurs

# Create a DataFrame with the collected titles, categories, and prices (up to 100)
df_book_data = pd.DataFrame(all_books_data[:100])
display(df_book_data)

Scraping catalogue page: http://books.toscrape.com/catalogue/page-1.html
  Scraping book detail: http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html
  Scraping book detail: http://books.toscrape.com/catalogue/tipping-the-velvet_999/index.html
  Scraping book detail: http://books.toscrape.com/catalogue/soumission_998/index.html
  Scraping book detail: http://books.toscrape.com/catalogue/sharp-objects_997/index.html
  Scraping book detail: http://books.toscrape.com/catalogue/sapiens-a-brief-history-of-humankind_996/index.html
  Scraping book detail: http://books.toscrape.com/catalogue/the-requiem-red_995/index.html
  Scraping book detail: http://books.toscrape.com/catalogue/the-dirty-little-secrets-of-getting-your-dream-job_994/index.html
  Scraping book detail: http://books.toscrape.com/catalogue/the-coming-woman-a-novel-based-on-the-life-of-the-infamous-feminist-victoria-woodhull_993/index.html
  Scraping book detail: http://books.toscrape.com/catalogue/the-boys-in

,Book Title,Category,Price
0,A Light in the Attic,Poetry,51.77
1,Tipping the Velvet,Historical Fiction,53.74
2,Soumission,Fiction,50.10
3,Sharp Objects,Mystery,47.82
4,Sapiens: A Brief History of Humankind,History,54.23
...,...,...,...
95,Lumberjanes Vol. 3: A Terrible Plan (Lumberjan...,Sequential Art,19.92
96,"Layered: Baking, Building, and Styling Spectac...",Food and Drink,40.11
97,Judo: Seven Steps to Black Belt (an Introducto...,Add a comment,53.90
98,Join,Science Fiction,35.67


In [32]:
# Sort the DataFrame by 'Price' in descending order and get the top 10 books
top_10_expensive_books = df_book_data.sort_values(by='Price', ascending=False).head(10)

display(top_10_expensive_books)

,Book Title,Category,Price
68,The Death of Humanity: and the Case for Life,Philosophy,58.11
40,Slow States of Collapse: Poems,Poetry,57.31
15,Our Band Could Be Your Life: Scenes from the A...,Music,57.25
58,The Past Never Ends,Mystery,56.50
57,The Pioneer Woman Cooks: Dinnertime: Comfort C...,Food and Drink,56.41
91,Masks and Shadows,Fantasy,56.40
56,The Secret of Dreadwillow Carse,Childrens,56.13
67,The Electric Pencil: Drawings from Inside Stat...,Nonfiction,56.06
25,Birdsong: A Story in Pictures,Childrens,54.64
4,Sapiens: A Brief History of Humankind,History,54.23
